# 第22章　学習ループを完全実装する ― 実戦仕様の骨格**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## 22.1　実戦仕様の学習ループ

In [ ]:
import torch, copy, mathfrom torch.amp import autocast, GradScalerdef train(model, train_loader, val_loader, evaluate, device,          epochs=100, lr=1e-4, patience=15, ckpt="best.pth"):    model = model.to(device)    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)    criterion = torch.nn.CrossEntropyLoss()    dev = torch.device(device).type                 # "cuda" か "cpu" か    amp_on = (dev == "cuda")                        # CPUでは混合精度を切る（警告の山を防ぐ）    scaler = GradScaler(dev, enabled=amp_on)    # 最良スコアは -inf から始める。0.0 で始めると、スコアが0のときや負の指標を    # 使うとき、best_state が初期重みのまま「学習済み」として返る。    # best_state は初期重みで埋めておき、load_state_dict(None) でのクラッシュも防ぐ。    best_score, best_state, no_improve = float("-inf"), copy.deepcopy(model.state_dict()), 0    got_valid_score = False        # 一度も有効な評価が得られなければ、明示的に失敗させる    for epoch in range(epochs):        # --- 学習フェーズ ---        model.train()        for x, y in train_loader:            x, y = x.to(device), y.to(device)            optimizer.zero_grad()            with autocast(dev, enabled=amp_on):        # 混合精度（CPUでは自動的に無効）                loss = criterion(model(x), y)            scaler.scale(loss).backward()            scaler.step(optimizer); scaler.update()        cur_lr = optimizer.param_groups[0]["lr"]        # このエポックで実際に使った学習率        scheduler.step()                                # 学習率を更新（次エポックぶん）        # --- 検証フェーズ ---        m = evaluate(model, val_loader, device)         # 13章と同じ契約（指標の辞書）        score = m[SELECT_KEY]                           # APなど退化に強い指標で選ぶ        if not math.isfinite(score):                    # 評価不能をスコア0として扱わない            print(f"epoch {epoch}: 評価不能（{SELECT_KEY} が有限値でない）")            continue        got_valid_score = True        print(f"epoch {epoch}: val {SELECT_KEY}={score:.4f} lr={cur_lr:.2e}")        # --- 最良の保存と早期終了 ---        if score > best_score:            best_score = score            best_state = copy.deepcopy(model.state_dict())            torch.save({"epoch": epoch, "model": best_state,                        "score": best_score}, ckpt)     # チェックポイント            no_improve = 0        else:            no_improve += 1            if no_improve >= patience:                  # 早期終了                print(f"早期終了（{patience}エポック改善なし）")                break    if not got_valid_score:                             # 静かに初期重みを返さない        raise RuntimeError("有効な検証スコアが一度も得られなかった")    model.load_state_dict(best_state)                   # 最良を復元して返す    return model, best_score

## 22.3　中断からの再開

In [ ]:
ckpt = torch.load("best.pth", map_location=device)model.load_state_dict(ckpt["model"])start_epoch = ckpt["epoch"] + 1              # 続きからprint(f"epoch {start_epoch} から再開（前回スコア {ckpt['score']:.4f}）")

## チェックポイントをGoogle Driveに残す ― 切断されても学習を無駄にしない

In [ ]:
from google.colab import driveimport osdrive.mount("/content/drive")                       # 初回は認証を求められるCKPT_DIR = "/content/drive/MyDrive/medai_ckpt"      # ここは再起動後も残るos.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import torchdef save_ckpt(state, path):    tmp = path + ".tmp"    torch.save(state, tmp)          # まず一時ファイルへ完全に書く    os.replace(tmp, path)           # 書き終えてから一気に置き換える（半端な上書きを残さない）

In [ ]:
state = {    "epoch": epoch,    "model": model.state_dict(),    "optimizer": optimizer.state_dict(),   # 欠くと再開後に学習率・モメンタムが巻き戻る    "scheduler": scheduler.state_dict(),    "best_score": best_score,    "classes": train_ds.classes,           # 予測番号→ラベル名（後で推論するのに必須）    "img_size": 224,    "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},}# 順序が大事。先に最良を判定して履歴を更新し、そのうえで last を保存する。# 逆にすると、last には「更新前の best_score」が入り、再開時に最良判定が巻き戻る。if score > best_score:    best_score, no_improve = score, 0    state["best_score"] = best_score    save_ckpt(state, f"{CKPT_DIR}/best.pth")                   # 改善時だけ（本番用）else:    no_improve += 1state["best_score"], state["no_improve"] = best_score, no_improvestate["scaler"] = scaler.state_dict()                          # AMPのスケーラも残すsave_ckpt(state, f"{CKPT_DIR}/last.pth")                       # 毎エポック（再開用）

In [ ]:
start_epoch = 0path = f"{CKPT_DIR}/last.pth"if os.path.exists(path):    ck = torch.load(path, map_location=device, weights_only=False)    model.load_state_dict(ck["model"])    optimizer.load_state_dict(ck["optimizer"])    scheduler.load_state_dict(ck["scheduler"])    start_epoch = ck["epoch"] + 1    best_score  = ck["best_score"]        # 表示するだけでなく、変数へ戻す    no_improve  = ck.get("no_improve", 0) # 早期終了のカウンタも引き継ぐ    if "scaler" in ck: scaler.load_state_dict(ck["scaler"])   # AMPのスケーラ    print(f"Driveの{path} から epoch {start_epoch} で再開（前回best {best_score:.4f}）")for epoch in range(start_epoch, epochs):    ...                                                        # 本章の train ループ本体

## 学習を「見ながら」回す ― TensorBoardとmatplotlibでの監視

In [ ]:
from torch.utils.tensorboard import SummaryWriterwriter = SummaryWriter("runs/exp1")          # ログはこのフォルダに溜まる# --- 学習ループ内、各エポック末に ---# loss.item() は「最終バッチの損失」。エポックの代表値にするなら、学習フェーズで#   epoch_loss_sum += loss.item() * x.size(0);  n_samples += x.size(0)# と件数で重み付けて足しておき、ここで割る。writer.add_scalar("loss/train", epoch_loss_sum / n_samples, epoch)writer.add_scalar(f"{SELECT_KEY}/val", score, epoch)   # 指標名は実際の中身に合わせるwriter.add_scalar("lr", cur_lr, epoch)                 # このエポックで実際に使った学習率writer.flush()                                # Driveに置くなら runs/ ごと保存すれば消えない

```text%load_ext tensorboard%tensorboard --logdir runs```

In [ ]:
from IPython.display import clear_outputimport matplotlib.pyplot as plthist = {"train_loss": [], "val": []}def plot_live(hist):    clear_output(wait=True)                  # 前の図を消して描き直す    plt.plot(hist["train_loss"], label="train loss")    plt.plot(hist["val"],        label=f"val {SELECT_KEY}")   # 凡例は記録している指標名に連動させる    plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=.3); plt.show()# 各エポック末: hist["train_loss"].append(epoch_loss_sum / n_samples); hist["val"].append(score); plot_live(hist)